# NYC Yellow Taxi Fare Prediction: Linear and Logistic Models (Apurbo)

This notebook runs the linear models on the same trips and the same chronological train / validation / test split as Sean's boosting notebook.

**What it trains** (each one twice: *Pre-trip* and *Post-trip*)
- Dollar fare: Linear Regression, Ridge (L2), Lasso (L1), and a median baseline
- Yes/no: Logistic Regression (L1 and L2) for "is the fare $35 or more?" (optional, can be switched off)

**Pre-trip** uses pickup zone, drop-off zone and pickup time (what a rider knows in advance).
**Post-trip** adds actual distance and duration (known only after the ride).

All the real code lives in `linear_models.py` in the repo root. This notebook just calls it, so the notebook and the script always give the same answers.

## Step 1: Setup
On Google Colab this downloads the repo. On your own computer it finds the repo root automatically.

In [ ]:
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("MIS545-NYCTaxi").exists():
        subprocess.run(["git", "clone", "https://github.com/nataperezzr/MIS545-NYCTaxi.git"], check=True)
    os.chdir("MIS545-NYCTaxi")
else:
    p = Path.cwd()
    while p != p.parent and not (p / "linear_models.py").exists():
        p = p.parent
    os.chdir(p)
sys.path.insert(0, os.getcwd())
print("Working in:", os.getcwd())

import warnings
import pandas as pd
import linear_models as lm
warnings.filterwarnings("ignore", category=UserWarning)

## Step 2: Settings
Change these, then run the cells below.

In [ ]:
DATA_PATH = "data/yellow_tripdata_2026-05.parquet"
SAMPLE_ROWS = 100_000     # 0 = use the whole file (slower, needs more memory)
FARE_CUTOFF = 35.0        # dollars, for the yes/no label
RUN_LOGISTIC = True       # set False to skip logistic regression
CARD_ONLY = False         # True = card payments and passenger_count > 0 only (outline says this; Sean's notebook does not)
USE_SYNTHETIC = False     # True = fake demo data, only to test that the notebook runs

## Step 3: Load the data
If the parquet file is not in `data/`, the next cell downloads it from the NYC TLC (about 70-80 MB). On Colab, if the download is blocked, it asks you to upload the file.

In [ ]:
if USE_SYNTHETIC:
    print("Using SYNTHETIC demo data. Results are not real.")
    raw = lm.make_synthetic()
else:
    if not Path(DATA_PATH).exists():
        try:
            lm.download_data(DATA_PATH)
        except Exception as err:
            if "google.colab" not in sys.modules:
                raise SystemExit(f"Could not download the file ({err}). Put yellow_tripdata_2026-05.parquet in data/ and re-run.")
            from google.colab import files
            uploaded = files.upload()
            Path(DATA_PATH).parent.mkdir(exist_ok=True)
            os.replace(next(iter(uploaded)), DATA_PATH)
    raw = lm.load_data(DATA_PATH, SAMPLE_ROWS)
display(raw.head())

## Step 4: Clean the data
Same rules as Sean's notebook (positive fare/distance, 1 to 240 minutes, valid zones) so the rows match.

In [ ]:
clean = lm.clean_data(raw, card_only=CARD_ONLY)

## Step 5: Train, tune and evaluate
This one cell does everything: builds the features, splits the trips by time (70% / 15% / 15%), picks Ridge/Lasso `alpha` and logistic `C` using the **validation** period only, then scores once on the **test** period. It also runs 5-fold cross-validation and draws the learning curves, predicted-vs-actual plots and coefficient plots. Plots are also saved in `results/plots/`.

In [ ]:
reg_df, log_df, coefs = lm.run_experiment(
    clean, cutoff=FARE_CUTOFF, run_logistic=RUN_LOGISTIC, out_dir="results", show_plots=True)

## Step 6: Results
### Dollar-fare models (lower MAE / RMSE is better, higher R² is better)
The last four columns turn each predicted fare into a "$35 or more?" answer, so these models can be compared with the logistic model and with Sean's and Zhanel's models.

In [ ]:
display(reg_df.round(3))

### Logistic regression: is the fare $35 or more?
Class weights are balanced because $35+ trips are rare, so look at precision, recall, F1 and ROC-AUC rather than accuracy. (Empty if `RUN_LOGISTIC = False`, or if there are too few $35+ trips in the sample.)

In [ ]:
if len(log_df):
    display(log_df.drop(columns=["MAE ($)", "RMSE ($)", "R2"]).round(3))
else:
    print("No logistic results.")

### Biggest coefficients
Which inputs push the predicted fare up or down the most. Numeric inputs are scaled (per 1 standard deviation); zones and hours are compared with a baseline category.

In [ ]:
for (kind, approach), table in coefs.items():
    print(f"\n{kind} | {approach}")
    display(table.round(3))

### Learning curve numbers (Ridge)
If training error and validation error are both high and close together, the model is too simple (high bias). A big gap means overfitting.

In [ ]:
for approach in ["pretrip", "posttrip"]:
    print(approach)
    display(pd.read_csv(f"results/learning_curve_{approach}.csv").round(3))

## Using the $35 scale for the other models
Any model that predicts a dollar fare can be scored the same way, without retraining:

```python
from linear_models import threshold_metrics
threshold_metrics(y_test, predicted_fares, cutoff=35)
# -> Precision, Recall, F1, ROC-AUC
```

## Limits
- With `SAMPLE_ROWS = 100_000` the trips cover less than one day, so the numbers are not representative. Set `SAMPLE_ROWS = 0` for the whole month.
- One month of data says nothing about other seasons.
- Post-trip inputs (distance, duration) are not available for a quote before the ride.